In [ ]:
# ── Cell 0: install + auth ────────────────────────────────────────────────
!pip install unsloth trl peft transformers datasets wandb python-dotenv -q

import os
from huggingface_hub import login

# Secrets are added via Kaggle Notebook → Add-ons → Secrets
# Required secrets: HF_TOKEN, WANDB_API_KEY
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")

login(token=HF_TOKEN)  # required for gated meta-llama/Llama-3.1-8B-Instruct
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "tweet-scorer-finetuning"
print("Auth complete.")

In [ ]:
# ── Cell 1: load base model + tokenizer ──────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    dtype=None,       # auto: bfloat16 on Ampere+, float16 on P100
    load_in_4bit=True,
)
print("Base model loaded.")

In [ ]:
# ── Cell 2: apply QLoRA adapter ───────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves ~40% VRAM vs standard
    random_state=42,
)
print("LoRA adapter applied.")

In [ ]:
# ── Cell 3: load dataset + apply chat template ────────────────────────────
# The 'messages' column is a list of role/content dicts (not a plain string).
# SFTTrainer expects a string column, so we pre-map using the tokenizer's
# Llama-3.1 chat template. add_generation_prompt=False keeps the full
# assistant turn in the sequence (training target, not inference prompt).
from datasets import load_dataset

dataset = load_dataset("sud1157/tweet-scorer-dataset")
print(dataset)

def apply_chat_template(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True, remove_columns=["messages"])
print("Dataset formatted. Sample:")
print(dataset["train"][0]["text"][:300])

In [ ]:
# ── Cell 4: train ─────────────────────────────────────────────────────────
import wandb
from trl import SFTTrainer, SFTConfig

wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(project="tweet-scorer-finetuning", name="llama3-8b-qlora-r16")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir="/kaggle/working/tweet-scorer",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        warmup_ratio=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=True,   # P100 does not support bf16
        bf16=False,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="wandb",
        dataset_text_field="text",   # plain-string column after chat template
        max_seq_length=2048,
        packing=True,                # packs short examples → fewer wasted tokens
        dataset_num_proc=2,
    ),
)

trainer.train()
wandb.finish()
print("Training complete.")

In [ ]:
# ── Cell 5: push adapter to HF Hub ────────────────────────────────────────
model.push_to_hub("sud1157/tweet-scorer-llama3-8b", token=HF_TOKEN)
tokenizer.push_to_hub("sud1157/tweet-scorer-llama3-8b", token=HF_TOKEN)
print("Adapter pushed to https://huggingface.co/sud1157/tweet-scorer-llama3-8b")